# Food Lens — 0-batch: dataset audit (read-only)

This notebook **only reads** `MyDrive/food101_dataset`. It never writes, moves, renames or deletes anything on Drive.
All results go to the local Colab disk (`/content/food_lens_audit`), which disappears when the runtime ends.

What it checks:
1. File counts per folder and per target class (5 classes; `plov`/`pilaf` excluded).
2. Full decode of every image (truncated/corrupt files are reported).
3. Real format vs file extension, image size, tiny/animated images, EXIF orientation.
4. Exact duplicates (SHA-256), within and across classes.
5. Near-duplicates (perceptual hash), within and across classes.
6. A before/after listing of the dataset to prove nothing on Drive changed.

**Runtime:** CPU is enough — no GPU needed. Run the cells top to bottom.

In [ ]:
# 1) Settings — change only if your Drive folder has a different name.
DATASET_DIR = "/content/drive/MyDrive/food101_dataset"
OUT_DIR = "/content/food_lens_audit"  # local Colab disk, never Drive

# Source folder -> target class for the 5-class Food Lens scope.
TARGET_MAP = {
    "hamburger": "hamburger",
    "hot_dog": "hot_dog",
    "chocolate_cake": "dessert",
    "cheesecake": "dessert",
    "kebab": "kebab",
    "pizza": "pizza",
}
EXCLUDED_FOLDERS = {"plov", "pilaf"}  # out of scope; still counted for the record

NEAR_DUP_MAX_DISTANCE = 6  # perceptual-hash Hamming distance (0..64); lower = stricter
TINY_SIDE = 64             # images with a shorter side below this are flagged
TINY_BYTES = 2048          # files smaller than this are flagged
WORKERS = 16               # parallel readers (Drive is network storage)

In [ ]:
# 2) Mount Google Drive (Colab will ask you to allow access).
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3) Helpers. Everything below only opens dataset files in read mode ("rb").
import csv, hashlib, io, json, os, re, time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from PIL import Image, ImageOps

assert not os.path.abspath(OUT_DIR).startswith("/content/drive"), "OUT_DIR must not be on Drive"
assert os.path.isdir(DATASET_DIR), f"Dataset folder not found: {DATASET_DIR}"
os.makedirs(OUT_DIR, exist_ok=True)
Image.MAX_IMAGE_PIXELS = 120_000_000  # report huge images instead of failing on Pillow's default warning


def snapshot(root):
    """Path -> (size, mtime_ns) for every file under root. Used to prove the audit changed nothing."""
    result = {}
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        for name in filenames:
            path = os.path.join(dirpath, name)
            st = os.stat(path)
            result[os.path.relpath(path, root)] = (st.st_size, st.st_mtime_ns)
    return result


def sniff(head):
    """Real format from magic bytes, independent of the file extension."""
    if head.startswith(b"\xff\xd8\xff"):
        return "JPEG"
    if head.startswith(b"\x89PNG\r\n\x1a\n"):
        return "PNG"
    if head[:6] in (b"GIF87a", b"GIF89a"):
        return "GIF"
    if head[:4] == b"RIFF" and head[8:12] == b"WEBP":
        return "WEBP"
    if head[:2] == b"BM":
        return "BMP"
    if head[:4] in (b"II*\x00", b"MM\x00*"):
        return "TIFF"
    if head[4:8] == b"ftyp":
        return "HEIF/AVIF/MP4"
    if head.lstrip()[:1] == b"<":
        return "HTML/XML"
    return "unknown"


_N = 32
_k = np.arange(_N)[:, None]
_n = np.arange(_N)[None, :]
_DCT = np.cos(np.pi * (2 * _n + 1) * _k / (2 * _N)) * np.sqrt(2 / _N)
_DCT[0] /= np.sqrt(2)


def phash64(rgb):
    """64-bit perceptual hash: low-frequency DCT of a 32x32 grayscale image vs its median."""
    gray = np.asarray(rgb.convert("L").resize((_N, _N), Image.LANCZOS), dtype=np.float64)
    low = (_DCT @ gray @ _DCT.T)[:8, :8].flatten()
    bits = low > np.median(low[1:])
    return int("".join("1" if b else "0" for b in bits), 2)


EXT_FORMATS = {".jpg": "JPEG", ".jpeg": "JPEG", ".png": "PNG", ".webp": "WEBP", ".gif": "GIF", ".bmp": "BMP"}
TRAINING_EXTS = (".jpg", ".jpeg", ".png")  # extension filter used by the old menu_detector notebook
FOOD101_NAME = re.compile(r"^\d+\.jpg$")


def analyze(path):
    parent = os.path.dirname(path)
    folder = "(top level)" if os.path.samefile(parent, DATASET_DIR) else os.path.basename(parent)
    name = os.path.basename(path)
    ext = os.path.splitext(name)[1].lower()
    rec = {
        "folder": folder, "name": name, "ext": ext,
        "target": TARGET_MAP.get(folder, "EXCLUDED" if folder in EXCLUDED_FOLDERS else "UNMAPPED"),
        "food101_style_name": bool(FOOD101_NAME.match(name)),
        "in_old_training_filter": name.endswith(TRAINING_EXTS),
        "bytes": None, "sha256": None, "magic": None, "pil_format": None, "mode": None,
        "width": None, "height": None, "frames": None, "exif_orientation": None,
        "decode_ok": False, "error": "", "phash": None,
    }
    try:
        with open(path, "rb") as f:
            data = f.read()
    except OSError as exc:
        rec["error"] = f"read: {type(exc).__name__}: {exc}"[:200]
        return rec
    rec["bytes"] = len(data)
    rec["sha256"] = hashlib.sha256(data).hexdigest()
    rec["magic"] = sniff(data[:16])
    try:
        with Image.open(io.BytesIO(data)) as im:
            rec["pil_format"] = im.format
            rec["mode"] = im.mode
            rec["width"], rec["height"] = im.size
            rec["frames"] = getattr(im, "n_frames", 1)
            im.load()  # full decode; truncated files raise here
            rec["exif_orientation"] = im.getexif().get(0x0112)
            rgb = ImageOps.exif_transpose(im).convert("RGB")
            rec["phash"] = phash64(rgb)
            rec["decode_ok"] = True
    except Exception as exc:  # report, never fix
        rec["error"] = f"decode: {type(exc).__name__}: " + re.sub(r" <[^>]*>", "", str(exc))[:160]
    return rec


print("Helpers ready. Output folder:", OUT_DIR)

In [ ]:
# 4) Scan every file (read-only). Drive is slow on first access: expect a few minutes for ~5,000 files.
t0 = time.time()
before = snapshot(DATASET_DIR)
folders = sorted(d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d)))
loose_files = sorted(f for f in os.listdir(DATASET_DIR) if os.path.isfile(os.path.join(DATASET_DIR, f)))
paths = [os.path.join(DATASET_DIR, rel) for rel in sorted(before)]
nested = [rel for rel in before if rel.count(os.sep) > 1]
print("Folders:", folders)
print("Files at top level (not in a class folder):", loose_files[:20])
print("Files nested deeper than one folder:", len(nested))
print("Total files:", len(paths))

with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    records = []
    for i, rec in enumerate(pool.map(analyze, paths), 1):
        records.append(rec)
        if i % 500 == 0:
            print(f"  {i}/{len(paths)} files, {time.time() - t0:.0f}s")
print(f"Scanned {len(records)} files in {time.time() - t0:.0f}s")

In [ ]:
# 5) Report: counts, decode, format, size.
def pct(values, q):
    return int(np.percentile(values, q)) if values else None

ok = [r for r in records if r["decode_ok"]]
by_folder = defaultdict(list)
for r in records:
    by_folder[r["folder"]].append(r)

folder_rows = []
for folder in sorted(by_folder):
    rs = by_folder[folder]
    good = [r for r in rs if r["decode_ok"]]
    short = [min(r["width"], r["height"]) for r in good]
    folder_rows.append({
        "folder": folder,
        "target": rs[0]["target"],
        "files": len(rs),
        "decoded": len(good),
        "failed": len(rs) - len(good),
        "ext_mismatch": sum(1 for r in rs if r["magic"] and EXT_FORMATS.get(r["ext"]) not in (None, r["magic"])),
        "unknown_ext": sum(1 for r in rs if r["ext"] not in EXT_FORMATS),
        "skipped_by_old_filter": sum(1 for r in rs if not r["in_old_training_filter"]),
        "tiny": sum(1 for r in rs if (r["bytes"] or 0) < TINY_BYTES
                    or (r["decode_ok"] and min(r["width"], r["height"]) < TINY_SIDE)),
        "animated": sum(1 for r in good if (r["frames"] or 1) > 1),
        "food101_names": sum(1 for r in rs if r["food101_style_name"]),
        "short_side_p5": pct(short, 5), "short_side_p50": pct(short, 50), "short_side_p95": pct(short, 95),
    })

magic_by_folder = {f: dict(Counter(r["magic"] for r in rs)) for f, rs in by_folder.items()}
errors = [r for r in records if not r["decode_ok"]]
mismatches = [r for r in records if r["magic"] and EXT_FORMATS.get(r["ext"]) not in (None, r["magic"])]

# Target-class view: decodable files, and unique by content (SHA-256).
target_rows = []
for target in sorted({r["target"] for r in records}):
    rs = [r for r in ok if r["target"] == target]
    target_rows.append({"target": target, "decoded": len(rs), "unique_sha256": len({r["sha256"] for r in rs})})


def table(rows):
    if not rows:
        return "_(none)_"
    cols = list(rows[0])
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join("" if row[c] is None else str(row[c]) for c in cols) + " |" for row in rows]
    return "\n".join(lines)


print(table(folder_rows))
print()
print(table(target_rows))
print("\nReal formats per folder:", json.dumps(magic_by_folder, indent=1))
print(f"\nDecode failures: {len(errors)}")
for r in errors[:30]:
    print(f"  {r['folder']}/{r['name']}: {r['error']}")
print(f"\nExtension vs real format mismatches: {len(mismatches)}")
for r in mismatches[:30]:
    print(f"  {r['folder']}/{r['name']}: extension {r['ext']} but content is {r['magic']}")

In [ ]:
# 6) Exact duplicates (identical bytes) and near-duplicates (perceptual hash).
by_sha = defaultdict(list)
for r in ok:
    by_sha[r["sha256"]].append(r)
exact_groups = [rs for rs in by_sha.values() if len(rs) > 1]
exact_cross = [rs for rs in exact_groups if len({r["target"] for r in rs}) > 1]
print(f"Exact duplicate groups: {len(exact_groups)} "
      f"({sum(len(g) - 1 for g in exact_groups)} redundant files); across target classes: {len(exact_cross)}")
for g in exact_cross[:20]:
    print("  CROSS:", [f"{r['folder']}/{r['name']}" for r in g])

# Near-duplicates: compare one representative per unique SHA-256.
reps = [rs[0] for rs in by_sha.values()]
hashes = np.array([r["phash"] for r in reps], dtype=np.uint64)
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
near_pairs = []
for start in range(0, len(reps), 256):
    block = hashes[start:start + 256, None] ^ hashes[None, :]
    dist = POPCOUNT[block.view(np.uint8)].reshape(block.shape[0], block.shape[1], 8).sum(axis=2)
    ii, jj = np.nonzero(dist <= NEAR_DUP_MAX_DISTANCE)
    for i, j in zip(ii + start, jj):
        if j > i:
            near_pairs.append((reps[i], reps[j], int(dist[i - start, j])))
near_cross = [p for p in near_pairs if p[0]["target"] != p[1]["target"]]
near_within = Counter(p[0]["target"] for p in near_pairs if p[0]["target"] == p[1]["target"])
print(f"\nNear-duplicate pairs (distance <= {NEAR_DUP_MAX_DISTANCE}): {len(near_pairs)}; "
      f"across target classes: {len(near_cross)}")
print("Within-class near-duplicate pairs:", dict(near_within))
for a, b, d in sorted(near_pairs, key=lambda p: p[2])[:30]:
    print(f"  d={d}: {a['folder']}/{a['name']}  <->  {b['folder']}/{b['name']}")
print("\nNote: near-duplicate = visually similar by hash. Confirm by eye (cell 9) before treating as duplicates.")

In [ ]:
# 7) Save results to the local Colab disk (not Drive) and print a summary to copy back.
after = snapshot(DATASET_DIR)
unchanged = before == after

files_csv = os.path.join(OUT_DIR, "files.csv")
with open(files_csv, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(records[0]))
    writer.writeheader()
    for r in records:
        writer.writerow({**r, "phash": "" if r["phash"] is None else f"{r['phash']:016x}"})

summary = {
    "dataset_dir": DATASET_DIR,
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "total_files": len(records),
    "decoded": len(ok),
    "decode_failures": len(errors),
    "ext_mismatches": len(mismatches),
    "folders": folder_rows,
    "targets": target_rows,
    "real_formats_per_folder": magic_by_folder,
    "exact_duplicate_groups": len(exact_groups),
    "exact_duplicate_redundant_files": sum(len(g) - 1 for g in exact_groups),
    "exact_duplicate_groups_cross_class": len(exact_cross),
    "near_duplicate_threshold": NEAR_DUP_MAX_DISTANCE,
    "near_duplicate_pairs": len(near_pairs),
    "near_duplicate_pairs_cross_class": len(near_cross),
    "near_duplicate_pairs_within_class": dict(near_within),
    "top_level_loose_files": loose_files,
    "nested_files": len(nested),
    "drive_unchanged_during_audit": unchanged,
}
with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

report = [
    "# Food Lens dataset audit",
    f"- Dataset: `{DATASET_DIR}`  ·  generated (UTC): {summary['generated_utc']}",
    f"- Files: {len(records)} · decoded: {len(ok)} · decode failures: {len(errors)} · ext mismatches: {len(mismatches)}",
    f"- Exact duplicate groups: {len(exact_groups)} (cross-class: {len(exact_cross)})",
    f"- Near-duplicate pairs (d<={NEAR_DUP_MAX_DISTANCE}): {len(near_pairs)} (cross-class: {len(near_cross)})",
    f"- Drive dataset unchanged during audit: **{unchanged}**",
    "", "## Per folder", table(folder_rows), "", "## Per target class", table(target_rows),
    "", "## Decode failures (first 30)", *[f"- {r['folder']}/{r['name']}: {r['error']}" for r in errors[:30]],
    "", "## Extension mismatches (first 30)", *[f"- {r['folder']}/{r['name']}: {r['ext']} -> {r['magic']}" for r in mismatches[:30]],
]
with open(os.path.join(OUT_DIR, "summary.md"), "w") as f:
    f.write("\n".join(report))

print("\n".join(report))
print("\nSaved:", sorted(os.listdir(OUT_DIR)))
assert unchanged, "Dataset listing changed during the audit — stop and report this."

In [ ]:
# 8) OPTIONAL — download the three result files to your computer (no images inside).
from google.colab import files
for name in ("summary.md", "summary.json", "files.csv"):
    files.download(os.path.join(OUT_DIR, name))

In [ ]:
# 9) OPTIONAL — look at suspicious files and near-duplicate pairs (display only, nothing saved).
import matplotlib.pyplot as plt

def show(items, title, limit=12):
    items = items[:limit]
    if not items:
        print(title, "— none")
        return
    fig, axes = plt.subplots(len(items), 2 if isinstance(items[0], tuple) else 1,
                             figsize=(6, 3 * len(items)), squeeze=False)
    for row, item in zip(axes, items):
        pair = item[:2] if isinstance(item, tuple) else (item,)
        for ax, r in zip(row, pair):
            ax.axis("off")
            try:
                with Image.open(os.path.join(DATASET_DIR, r["folder"], r["name"])) as im:
                    ax.imshow(ImageOps.exif_transpose(im).convert("RGB"))
            except Exception as exc:
                ax.text(0.1, 0.5, f"cannot show: {type(exc).__name__}")
            ax.set_title(f"{r['folder']}/{r['name']}", fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show(near_cross, "Near-duplicates across classes")
show(sorted(near_pairs, key=lambda p: p[2]), "Closest near-duplicate pairs")
show([r for r in ok if min(r["width"], r["height"]) < TINY_SIDE or r["bytes"] < TINY_BYTES], "Tiny images")
show([r for r in by_folder.get("kebab", []) if r["decode_ok"]], "Kebab sample (manual quality check)", limit=24)